In [1]:
from datetime import date
from collections import Counter, defaultdict
import warnings
import os
from itertools import combinations

from rdkit import Chem
import periodictable
import h5py
import numpy as np

import qcportal
from qcportal.external import scaffold
from qcportal.molecules import Molecule
from qcportal.singlepoint import SinglepointDriver, QCSpecification
from qcportal.optimization import OptimizationSpecification
from qcelemental.models.procedures import OptimizationProtocols
from qcelemental.physical_constants import constants

# Fix locale for PostgreSQL
os.environ['LC_ALL'] = 'en_US.UTF-8'
os.environ['LANG'] = 'en_US.UTF-8'

ADDRESS = "https://api.qcarchive.molssi.org:443"
#qc_client = qcportal.PortalClient(ADDRESS, cache_dir=".")
from qcfractal.snowflake import FractalSnowflake
# Disable compute workers since we're just creating the dataset, not running calculations
snowflake = FractalSnowflake(compute_workers=0)
client = snowflake.client()

In [2]:
#!aria2c "https://zenodo.org/records/21841427/files/structures_modelforge.hdf5?download=1"

## Helper Functions

In [3]:
def remove_extraneous_dimension(array):
    shape = list(np.shape(array))
    if 1 in shape:
        shape.remove(1)
    return np.array(array).reshape(shape)

def get_symbols(atomic_numbers):
    return [str(periodictable.elements[x])for x in remove_extraneous_dimension(atomic_numbers)]

def get_molecular_formula(atomic_numbers):
    return "".join([str(y) for x1, x2 in Counter(get_symbols(atomic_numbers)).items() for y in [x1, x2] if y != 1])

def get_molecular_weight(atomic_numbers):
    return sum(periodictable.elements[x].mass for x in remove_extraneous_dimension(atomic_numbers))


In [4]:
def apply_mapping(mapping, input_dict, index=0):

    output = defaultdict(dict)
    for key, value in mapping.items():
        if isinstance(value, str):
            data = input_dict[value]
            if not isinstance(data, str):
                if key == "geometry": # update number of frames
                    lx = np.shape(data)[0]
                
                if key != "geometry":
                    if isinstance(data, bytes):
                        data = data.decode('utf-8')
                        lx = None
                    else:
                        data = remove_extraneous_dimension(data)
                        lx = len(data)

                if lx is not None: # and len(np.shape(data)) > 1:
                    if len(data) == lx:
                        output[key] = data[index]
                        continue
                    else:
                        raise ValueError(f"Expected {lx} configuration, but {len(data)} are present")

            if isinstance(data, bytes):
                data = data.decode('utf-8')
            output[key] = data
        elif isinstance(value, tuple): # function, input pairs
            output[key] = value[0](*(input_dict[k2] for k2 in value[1:]))
        elif isinstance(value, list):
            output[key].update({k2: input_dict[k2] for k2 in value})
        elif isinstance(value, dict):
            output[key].update(apply_mapping(value, input_dict, index=0))
            
    return output
            
def convert_hdf5_group(hdf5_group):
    output = {}
    for key, value in hdf5_group.items():
        if isinstance(value, h5py.Group):
            output[key] = convert_hdf5_group(value)
        elif isinstance(value, h5py.Dataset):
            data = value[()]
            if isinstance(data, np.ndarray):
                output[key] = data
            elif isinstance(data, np.bytes_):
                output[key] = data.decode('utf-8')  # Convert to string
            else:
                output[key] = data.item() if isinstance(data, np.generic) else data  # Convert NumPy scalars
        else:
            output[key] = value

    return output

In [5]:
def find_metal_coordination_atoms(cmiles: str, metal_symbol: str) -> tuple[int, list[int]] | None:
    """Extract metal center and coordinating atoms from CMILES.
    
    Returns
    -------
    tuple or None
        (metal_idx, coord_indices) if metal found, else None
    """
    mol = Chem.MolFromSmiles(cmiles, sanitize=False)
    if mol is None:
        return None
    
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == metal_symbol:
            metal_idx = atom.GetIdx()
            coord_indices = [n.GetIdx() for n in atom.GetNeighbors()]
            return metal_idx, coord_indices

    return None


def generate_angle_freeze_constraints(metal_idx: int, coord_indices: list[int]) -> list[dict]:
    """Generate QCSubmit angle freeze constraints.
    
    Returns list of constraint dicts for all pairwise angles: 
    coord_atom_i - metal - coord_atom_j
    """
    constraints = []
    for idx1, idx2 in combinations(coord_indices, 2):
        constraints.append({
            "type": "angle",
            "indices": [idx1, metal_idx, idx2]
        })
    return constraints

## Assembled Dataset

In [6]:
dataset_name = "OpenFF Architector Methyl-Capped Metal Complexes Optimization Dataset v0.0"
tagline = "Metal complexes with methyl-capped ligands optimized with BP86/def2-TZVP and constrained metal coordination geometries."
description = ("""
This dataset was generated using [architector](https://github.com/lanl/Architector/tree/Secondary_Solvation_Shell), the
details of the HDF5 file can be found in the Zenodo record (https://zenodo.org/records/19372923). This dataset contains
389,480 unique systems/configurations below 980 Da using the same keys as in the HDF5 as entry labels. The molecules
are limited to containing transition metals Pd, Zn, Fe, Cu, Li, or Mg with methyl-capped ligands (electronically neutral
capping groups) and also only contain elements C, H, P, S, O, N, F, Cl, or Br with overall charges: {-1,0,+1}. The metal
coordination for each metal center is between 1 and 12. Each molecule was preprocessed using gfn2-xtb as implemented in
the architector package. This optimization dataset was then run with the BP86/def2-TZVP. Each configuration is reported
with the following properties: 'energy', 'gradient', 'dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices',
'lowdin_charges', 'dipole_polarizabilities', 'mulliken_charges'. These structures are loosely optimized with constrained
metal coordination angles.
""")

dataset = client.add_dataset( # https://docs.qcarchive.molssi.org/user_guide/qcportal_reference.html
    "optimization", # collection type
    dataset_name, # Dataset name
    tagline=tagline,
    description=description,
    tags=["openff"],
    provenance={
        "qcportal": qcportal.__version__,
    },
    default_tag="openff",
    extras={
        "submitter": "jaclark5",
        "creation_date": date.today(),
        'collection_type': 'OptimizationDataset',
        "long_description": description,
        'long_description_url': f'https://github.com/openforcefield/qca-dataset-submission/tree/master/submissions/2026-08-05-{dataset_name.replace(" ", "-")}',
        "short_description": tagline,
        "dataset_name": dataset_name,
    },
)

In [7]:
hdf5_mapping = {
    "symbols": (get_symbols, "atomic_numbers"), 
    "geometry": "geometry",
    "molecular_charge": "total_charge",
    "molecular_multiplicity": "spin_multiplicities",
    "identifiers": {"molecular_formula": (get_molecular_formula, "atomic_numbers"), "canonical_isomeric_explicit_hydrogen_mapped_smiles": "cmiles"},
    "extras": {'molecular_weight': (get_molecular_weight, "atomic_numbers"), "canonical_isomeric_explicit_hydrogen_mapped_smiles": "cmiles"},
}

elements, molecular_weights, charges, multiplicities, metals, coord_ns, ox_states = [], [], [], [], [], [], []
conformers = Counter()
count_molecules = 0

errors_struc = defaultdict(list)
errors_mult = []
errors_misc = defaultdict(lambda: defaultdict(list))
failed_metals = defaultdict(lambda: 0)

hdf5 = h5py.File(f"structures_modelforge.hdf5", 'r')
for ii, (label, mol_hdf5) in enumerate(hdf5.items()):
    mol_dict = convert_hdf5_group(mol_hdf5)
    lx = 1
    metals.append(mol_dict["metal"].decode('utf-8'))
    coord_ns.append(mol_dict["coordination_number"])
    ox_states.append(mol_dict["oxidation_state"])
    
    ## Decide to filter
    try:
        input = apply_mapping(hdf5_mapping, mol_dict, index=0)
        if len(input["symbols"]) != np.shape(input["geometry"])[0]:
            raise ValueError(f"Geometries don't match number of symbols: {len(input['symbols'])} != {np.shape(input['geometry'])[0]}")
    except Exception as e:
        raise ValueError("Did not extract molecule")
        #errors_struc[str(e)[:30]].append([label, 1, str(e)])
        #continue

    input["geometry"] *= 10 / constants.bohr2angstroms # Convert from nm to Bohr (a0)

    # Extract metal coordination for angle constraints
    constraint_string = None
    metal_sym = mol_dict["metal"].decode('utf-8')
    cmiles = input["extras"]['canonical_isomeric_explicit_hydrogen_mapped_smiles']
    if metal_sym and cmiles:
        metal_info = find_metal_coordination_atoms(cmiles, metal_sym)
        if metal_info:
            metal_idx, coord_indices = metal_info
            constraints_list = generate_angle_freeze_constraints(metal_idx, coord_indices)
            if len(coord_indices) > 1 and not constraints_list:
                raise ValueError("No constraints found")
        else:
            raise ValueError("Coordination atoms could not be found")
    else:
        raise ValueError(f"Could not find cmiles and metal: {metal_sym}, {cmiles}")

    try:
        molecule = Molecule(
            name=label,
            fix_com=True,
            fix_orientation=True,
            fix_symmetry="c1",
            **input
        )
        dataset.add_entry(
            name=label, 
            initial_molecule=molecule,
            additional_keywords={"constraints": {"freeze": constraints_list}},
        )
        count_molecules += 1
        conformers[label] += 1
    except Exception as e:
        if "Inconsistent or unspecified chg/mult" in str(e):
            errors_mult.append(label)
        else:
            errors_misc[str(e)[:30]][label].append(str(e))
        continue

    elements.extend(list(set(input['symbols'])))
    molecular_weights.append(input['extras']["molecular_weight"])
    charges.append(input["molecular_charge"])
    multiplicities.append(input["molecular_multiplicity"])

dataset.extras["elements"] = sorted(list(set(elements)))

Connection error for http://localhost:56849/api/v1/datasets/optimization/1/entries/bulkFetch: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkFetch HTTP/1.1\r\n')) - retrying in 0.49 seconds [1/5]
Connection error for http://localhost:56849/api/v1/datasets/optimization/1/entries/bulkFetch: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkFetch HTTP/1.1\r\n')) - retrying in 0.51 seconds [1/5]
Connection error for http://localhost:56849/api/v1/datasets/optimization/1/entries/bulkFetch: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkFetch HTTP/1.1\r\n')) - retrying in 0.52 seconds [1/5]
Connection error for http://localhost:56849/api/v1/datasets/optimization/1/entries/bulkFetch: ('Connection aborted.', BadStatusLine('POST /api/v1/datasets/optimization/1/entries/bulkFetch HTTP/1.1\r\n')) - retrying in 0.51 seconds [1/5]
Connection error for http://localhost:56849/api/v1/d

In [8]:
print(f"Number of molecules removed for unspecified chg/mult: {len(errors_mult)}")
print(f"Number of molecules removed for structure issues: {len(errors_struc)}")
print(f"Number of conformers accepted: {len(dataset.entry_names)}")

Number of molecules removed for unspecified chg/mult: 0
Number of molecules removed for structure issues: 0
Number of conformers accepted: 389480


In [9]:
print(f"{len(dataset.entry_names)} conformers were imported.")

print("\nThe following errors DO remove molecules from the dataset:")
for err, values in errors_misc.items():
    print(f"    {len(values)}: '{err}'")

389480 conformers were imported.

The following errors DO remove molecules from the dataset:


In [10]:

spec = QCSpecification(
    program='psi4',
    driver=SinglepointDriver.gradient,
    method='BP86',
    basis='def2-TZVP',
    keywords={
        'maxiter': 500, 
        'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'],
        'function_kwargs': {'properties': ['dipole_polarizabilities']},
        'reference': 'uks',
    },
    protocols={'wavefunction': 'none'}
)
opt_spec = OptimizationSpecification(
    program="geometric",
    qc_specification=spec, 
    keywords={
        "tmax": 0.3,
        "check": 0,
        "qccnv": False,
        "reset": True,
        "trust": 0.1,
        "molcnv": False,
        "enforce": 0.0,
        "epsilon": 1e-05,
        "maxiter": 300,
        "coordsys": "dlc",
        "convergence_set": "GAU",
        "converge": ['energy', '1e-3', 'grms', '0.2', 'gmax', '1.0', 'drms', '15', 'dmax', '30'],
    },
)
dataset.add_specification(name="BP86/def2-TZVP", specification=opt_spec)

InsertMetadata(error_description=None, errors=[], inserted_idx=[0], existing_idx=[])

In [11]:
scaffold.to_json(dataset, compress=True)
#dataset.submit()

## Make Outputs

In [12]:
print("Elements:", ", ".join(dataset.extras["elements"]))
print("Charges:", sorted(set([float(x) for x in charges])))
print("Multiplicities:", sorted(set([int(x) for x in multiplicities])))
print("Metals:", Counter(metals))
print("Coordination Numbers:", Counter(coord_ns))
print("Oxidation States:", Counter(ox_states))

print("Molecular Weight (min mean max):", int(np.min(molecular_weights)), int(np.mean(molecular_weights)), int(np.max(molecular_weights)))
            
print("Number of Molecules:", len(conformers))
print("Number of Conformers:", sum(conformers.values()))
n_conformers = np.array(list(conformers.values()))
print("Number of conformers (min mean max):", int(np.min(n_conformers)), int(np.mean(n_conformers)), int(np.max(n_conformers)))

Elements: Br, C, Cu, Fe, H, Li, Mg, N, O, P, Pd, S, Zn
Charges: [-1.0, 0.0, 1.0]
Multiplicities: [1, 2, 3, 4, 5, 6]
Metals: Counter({'Fe': 131503, 'Pd': 88048, 'Cu': 77034, 'Mg': 43192, 'Zn': 37551, 'Li': 12152})
Coordination Numbers: Counter({8: 93460, 7: 66692, 6: 66486, 5: 52250, 4: 44696, 9: 35468, 3: 17797, 10: 7720, 2: 2644, 12: 2081, 1: 186})
Oxidation States: Counter({2: 86821, 1: 85409, 4: 55195, 3: 54960, 5: 36500, 0: 35442, 6: 18118, 7: 17035})
Molecular Weight (min mean max): 22 434 980
Number of Molecules: 389480
Number of Conformers: 389480
Number of conformers (min mean max): 1 1 1


In [13]:
for spec, obj in dataset.specifications.items():
    obj = obj.dict()
    obj = obj['specification']
    print(obj.keys())
    print("* Spec:", spec)
    print(f"    * program: {obj['program']}")
    print(f"    * keywords:")
    for k, v in obj["keywords"].items():
        print(f"       * {k}: {v}")
    print(f"    * qc_specification:")
    for k, field in obj['qc_specification'].items():
        print(f"       * {k}: {field}")
    print("* SCF properties:")
    for field in obj['qc_specification']['keywords']["scf_properties"]:
        print(f"       * {field}")

dict_keys(['program', 'qc_specification', 'keywords', 'protocols'])
* Spec: BP86/def2-TZVP
    * program: geometric
    * keywords:
       * tmax: 0.3
       * check: 0
       * qccnv: False
       * reset: True
       * trust: 0.1
       * molcnv: False
       * enforce: 0.0
       * epsilon: 1e-05
       * maxiter: 300
       * converge: ['energy', '1e-3', 'grms', '0.2', 'gmax', '1.0', 'drms', '15', 'dmax', '30']
       * coordsys: dlc
       * convergence_set: GAU
    * qc_specification:
       * program: psi4
       * driver: SinglepointDriver.deferred
       * method: bp86
       * basis: def2-tzvp
       * keywords: {'maxiter': 500, 'reference': 'uks', 'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 'lowdin_charges', 'mulliken_charges'], 'function_kwargs': {'properties': ['dipole_polarizabilities']}}
       * protocols: {'wavefunction': <WavefunctionProtocolEnum.none: 'none'>, 'stdout': True, 'error_correction': {'default_policy': True, 'polic